# Context Engineering: Complete Guide

Build adaptive agents using all four context engineering strategies:
1. **Selecting Context** - Dynamic prompts from runtime context & store
2. **Writing Context** - Learn preferences from user feedback
3. **Summarizing Context** - Manage long conversations efficiently
4. **Isolating Context** - Delegate to specialized subagents

## Setup

In [1]:
from dotenv import load_dotenv
from datetime import datetime
from dataclasses import dataclass

load_dotenv()

from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore

checkpointer = InMemorySaver()
store = InMemoryStore()

In [2]:
@dataclass
class EmailAssistantContext:
    user_name: str
    timezone: str

context = EmailAssistantContext(
    user_name="Sydney",
    timezone="America/New_York"
)

## Define Tools

In [3]:
from langchain.tools import tool, ToolRuntime

# Main agent tools
@tool
def send_email(subject: str, content: str, runtime: ToolRuntime[EmailAssistantContext]) -> str:
    """Send email reply.
    
    Args:
        subject: Subject line
        content: Email body
    """
    return f"✅ Email sent from {runtime.context.user_name}!\n\nSubject: {subject}\n\n{content}"

## Strategy 1: Selecting Context

Create a dynamic prompt that reads from runtime context and store.

In [4]:
from langchain.agents.middleware import dynamic_prompt, ModelRequest

@dynamic_prompt
def personalized_prompt(request: ModelRequest) -> str:
    """Build system prompt from runtime context and store."""
    user = request.runtime.context.user_name
    tz = request.runtime.context.timezone
    time = datetime.now().strftime('%I:%M %p')
    
    # Read user preferences from store
    tone_pref = request.runtime.store.get(("prefs",), f"{user}/tone")
    tone = tone_pref.value if tone_pref else "professional"
    
    return f"""You are an email assistant for {user}.
Please sign all emails with the name {user}.

Time: {time} ({tz})
Tone: {tone}

Draft email replies using send_email tool."""

### Build Agent v1: Selecting Context Only

In [8]:
from langchain.agents import create_agent

agent_v1 = create_agent(
    model="openai:gpt-4o",
    tools=[send_email],
    middleware=[personalized_prompt],
    context_schema=EmailAssistantContext,
    store=store,
    name="agent_v1"
)

### Test Strategy 1: Dynamic Prompts

In [9]:
from utils import format_email

email1 = format_email(
    from_addr="alice@company.com",
    subject="Q4 Report",
    body="Hi Sydney, do you have the Q4 report ready? Thanks, Alice"
)

result = agent_v1.invoke({"messages": [{"role": "user", "content": f"Respond to: {email1}"}]}, context=context)
for msg in result["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Respond to: **From:** alice@company.com
**Subject:** Q4 Report

Hi Sydney, do you have the Q4 report ready? Thanks, Alice
================================== Ai Message ==================================
Tool Calls:
  send_email (call_pT8SUOKSpEaD8r336T4gDYad)
 Call ID: call_pT8SUOKSpEaD8r336T4gDYad
  Args:
    subject: Re: Q4 Report
    content: Hi Alice,

I am in the final stages of reviewing the Q4 report and ensuring all details are accurate. I will have it ready for you by the end of the day. Please let me know if there are any specific sections you would like me to focus on or any additional information you need.

Best regards,

Sydney
================================= Tool Message =================================
Name: send_email

✅ Email sent from Sydney!

Subject: Re: Q4 Report

Hi Alice,

I am in the final stages of reviewing the Q4 report and ensuring all details are accurate. I will have it re

### change tone manually

In [10]:
store.put(("prefs",), f"{context.user_name}/tone", "casual")

In [11]:
result = agent_v1.invoke({"messages": [{"role": "user", "content": f"Respond to: {email1}"}]}, context=context)
for msg in result["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Respond to: **From:** alice@company.com
**Subject:** Q4 Report

Hi Sydney, do you have the Q4 report ready? Thanks, Alice
================================== Ai Message ==================================
Tool Calls:
  send_email (call_0is3D06OFqHzO1ZlWfpEL1mn)
 Call ID: call_0is3D06OFqHzO1ZlWfpEL1mn
  Args:
    subject: Re: Q4 Report
    content: Hi Alice,

I’m finishing up the last sections of the Q4 report. I'll have it ready for you by the end of the day. Let me know if there's anything specific you’d like me to focus on.

Thanks,
Sydney
================================= Tool Message =================================
Name: send_email

✅ Email sent from Sydney!

Subject: Re: Q4 Report

Hi Alice,

I’m finishing up the last sections of the Q4 report. I'll have it ready for you by the end of the day. Let me know if there's anything specific you’d like me to focus on.

Thanks,
Sydney
========================

## Strategy 2: Writing Context

Add a hook that interrupts for approval and learns from user edits.

In [12]:
from langchain.agents.middleware import after_model, AgentState
from langgraph.runtime import Runtime
from langgraph.types import interrupt
from typing import Any

from utils import create_interrupt_request, create_review_config

llm = ChatOpenAI(model="gpt-4o")


def learn_from_edit(original: dict, edited: dict, runtime: Runtime) -> None:
    """Use LLM to extract preferences from a single edit."""
    user = runtime.context.user_name
    
    # Single LLM call to analyze edit and extract tone preference
    analysis_prompt = f"""Compare these two email drafts and determine the user's preferred tone.

ORIGINAL:
Subject: {original.get('subject', '')}
Body: {original.get('content', '')}

EDITED:
Subject: {edited.get('subject', '')}
Body: {edited.get('content', '')}

Based on the changes (length, formality, word choice, greetings/closings), what tone does the user prefer?

Respond with ONLY ONE WORD: "casual" or "professional" """

    response = llm.invoke([{"role": "user", "content": analysis_prompt}])
    tone_pref = response.content.strip().lower()
    
    if tone_pref in ["casual", "professional"]:
        runtime.store.put(("prefs",), f"{user}/tone", tone_pref)
        print(f"📚 Learned preference: {tone_pref} tone")


@after_model
def review_and_learn_hook(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Show draft for review using HITL syntax. Learn from edits using LLM."""
    
    messages = state.get("messages", [])
    if not messages:
        return None
    
    response = messages[-1]
    
    # Only interrupt for send_email tool calls
    if not hasattr(response, 'tool_calls') or not response.tool_calls:
        return None
    
    # Filter for send_email calls
    send_email_calls = [tc for tc in response.tool_calls if tc["name"] == "send_email"]
    
    if not send_email_calls:
        return None
    
    # Create HITL request using new syntax (helper functions in utils.py)
    action_requests = [create_interrupt_request(tc) for tc in send_email_calls]
    review_configs = [create_review_config(tc["name"]) for tc in send_email_calls]
    
    hitl_request = {
        "action_requests": action_requests,
        "review_configs": review_configs
    }
    
    # Show draft and wait for decisions
    hitl_response = interrupt(hitl_request)
    decisions = hitl_response["decisions"]
    
    # Process decisions and learn from edits
    for i, decision in enumerate(decisions):
        if decision["type"] == "edit":
            original_args = send_email_calls[i]["args"]
            edited_action = decision["edited_action"]
            edited_args = edited_action["args"]
            
            # Learn from the edit
            learn_from_edit(original_args, edited_args, runtime)
            
            # Update the tool call with edited version
            original_index = response.tool_calls.index(send_email_calls[i])
            response.tool_calls[original_index]["args"] = edited_args
    
    return None

### Build Agent v2: Selecting + Writing Context

In [13]:
agent_v2 = create_agent(
    model="openai:gpt-4o",
    tools=[send_email],
    middleware=[personalized_prompt, review_and_learn_hook],
    context_schema=EmailAssistantContext,
    store=store,
    checkpointer=checkpointer,
    name="agent_v2"
)

### Test Strategy 2: Learning from Edits

In [14]:
from langgraph.types import Command
from utils import create_review_ui

# Example: Create an interactive UI for reviewing a draft
email2 = format_email(
    from_addr="bob@company.com",
    subject="Coffee?",
    body="Want to grab coffee next week?"
)

config_v2 = {"configurable": {"thread_id": "v2-interactive"}}

# Start the agent - it will create a draft and pause
result = agent_v2.invoke(
    {"messages": [{"role": "user", "content": f"Respond to: {email2}"}]}, 
    config=config_v2, 
    context=context
)

# Get the draft from the interrupt
if result.get("__interrupt__"):
    interrupt_data = result["__interrupt__"][0].value
    action_request = interrupt_data["action_requests"][0]
    
    # Create and display interactive UI
    ui, get_decision = create_review_ui(
        draft_subject=action_request["args"]["subject"],
        draft_content=action_request["args"]["content"]
    )
    
    display(ui)
    
    # Wait for user decision (blocks until button is clicked)
    # Uncomment to use:
    decision = get_decision()
    final_result = agent_v2.invoke(
        Command(resume={"decisions": [decision]}),
        config=config_v2,
        context=context
    )

    for msg in final_result["messages"]:
        msg.pretty_print()

📚 Learned preference: casual tone
================================ Human Message =================================

Respond to: **From:** bob@company.com
**Subject:** Coffee?

Want to grab coffee next week?
================================== Ai Message ==================================
Tool Calls:
  send_email (call_8TzSSsiE61NifhOUEI4U7nrz)
 Call ID: call_8TzSSsiE61NifhOUEI4U7nrz
  Args:
    subject: Re: Coffee?
    content: Hey bob, how's 8am on Friday?

Sydney
================================= Tool Message =================================
Name: send_email

✅ Email sent from Sydney!

Subject: Re: Coffee?

Hey bob, how's 8am on Friday?

Sydney
================================== Ai Message ==================================

I've scheduled the coffee meeting for Friday at 8am. If there are any changes or further preferences, feel free to let me know! 

Sydney


In [25]:
from langgraph.types import Command

email2 = format_email(
    from_addr="bob@company.com",
    subject="Coffee?",
    body="Want to grab coffee next week?"
)

config_v2 = {"configurable": {"thread_id": "v2-test"}}

result = agent_v2.invoke({"messages": [{"role": "user", "content": f"Respond to: {email2}"}]}, config=config_v2, context=context)
for msg in result["messages"]:
    msg.pretty_print()

print("\n")
print(result["__interrupt__"][0].value['action_requests'][0]['description'])

================================ Human Message =================================

Respond to: **From:** bob@company.com
**Subject:** Coffee?

Want to grab coffee next week?
================================== Ai Message ==================================
Tool Calls:
  send_email (call_JymTsDMbYUqT5z45YMiEDUmc)
 Call ID: call_JymTsDMbYUqT5z45YMiEDUmc
  Args:
    subject: Re: Coffee?
    content: Hey Bob,

Coffee sounds great! How about we schedule it for sometime next week? Let me know what day works best for you.

Looking forward to it!

Cheers,
Sydney


📧 **Draft Email**

**Subject:** Re: Coffee?

**Body:**
Hey Bob,

Coffee sounds great! How about we schedule it for sometime next week? Let me know what day works best for you.

Looking forward to it!

Cheers,
Sydney

---
*Review the draft above and choose to approve or edit.*


In [26]:
final_result = agent_v2.invoke(
    Command(resume={
        "decisions": [{"type": "edit", "edited_action": {
            "args": {"subject": "8am coffee", "content": "I'm free at 8am on Friday, let's grab coffee then?"},
            "name": "send_email"
        }}]
    }), 
    config=config_v2,
    context=context
)

for msg in final_result["messages"]:
    msg.pretty_print()

📚 Learned preference: casual tone
================================ Human Message =================================

Respond to: **From:** bob@company.com
**Subject:** Coffee?

Want to grab coffee next week?
================================== Ai Message ==================================
Tool Calls:
  send_email (call_JymTsDMbYUqT5z45YMiEDUmc)
 Call ID: call_JymTsDMbYUqT5z45YMiEDUmc
  Args:
    subject: 8am coffee
    content: I'm free at 8am on Friday, let's grab coffee then?
================================= Tool Message =================================
Name: send_email

✅ Email sent from Sydney!

Subject: 8am coffee

I'm free at 8am on Friday, let's grab coffee then?
================================== Ai Message ==================================

I've sent an email suggesting meeting for coffee at 8am on Friday. Let me know if there's anything else you'd like to add or modify!

Sydney


## Strategy 3: Summarizing Context

Add summarization middleware to manage long conversations.

In [15]:
from langchain.agents.middleware import SummarizationMiddleware

summarization = SummarizationMiddleware(
    model="gpt-4o-mini",
    max_tokens_before_summary=500,
    messages_to_keep=4,
)

### Build Agent v3: Selecting + Writing + Summarizing

In [16]:
agent_v3 = create_agent(
    model="openai:gpt-4o",
    tools=[send_email],
    middleware=[personalized_prompt, summarization],
    context_schema=EmailAssistantContext,
    store=store,
    checkpointer=checkpointer,
    name="agent_v3"
)

### Test Strategy 3: Summarization with Email Chain

In [17]:
email_chain = [
    {"from": "product@company.com", "subject": "Product Launch",
     "body": "Can your team handle the documentation by March 15th?"},
    
    {"from": "marketing@company.com", "subject": "Marketing Materials",
     "body": "We need your input on the marketing copy."},
    
    {"from": "legal@company.com", "subject": "Legal Review",
     "body": "All docs need disclaimers by March 20th."},

    {"from": "ceo@company.com", "subject": "Status check",
     "body": "Can you give me a status update on all the documentation deliverables?"}
]

config_v3 = {"configurable": {"thread_id": "v3-test2"}}

for email_data in email_chain:
    email = format_email(
        from_addr=email_data["from"],
        subject=email_data["subject"],
        body=email_data["body"]
    )
    
    result = agent_v3.invoke({"messages": [{"role": "user", "content": f"Respond to: {email}"}]}, config_v3, context=context)

for msg in result["messages"]:
    msg.pretty_print()

print(f"\nFinal state: {len(result["messages"])} messages (summarization kept it manageable)")

================================ Human Message =================================

Here is a summary of the conversation to date:

- Responded to product@company.com confirming that the team can handle the documentation by March 15th.
- Responded to marketing@company.com expressing willingness to help with the marketing copy and asking when to discuss details.
- Responded to legal@company.com confirming all documents will have necessary disclaimers by March 20th.
================================== Ai Message ==================================

I've sent a reply confirming that we'll have all necessary disclaimers by March 20th. Let me know if there's anything else you need!
================================ Human Message =================================

Respond to: **From:** ceo@company.com
**Subject:** Status check

Can you give me a status update on all the documentation deliverables?
================================== Ai Message ==================================
Tool Calls:
  send_

## 5. Strategy 4: Isolating Context

Add a calendar subagent with specialized tools.

### More Tools

In [20]:
# Calendar subagent tools
@tool
def check_calendar(date: str, runtime: ToolRuntime[EmailAssistantContext]) -> str:
    """Check calendar for a date.
    
    Args:
        date: Date in YYYY-MM-DD format
    """
    return f"""Events on {date}:
- 9:00 AM: Standup (30 min)
- 2:00 PM: Client Meeting (1 hr)

Free: 10 AM-1 PM, 3-5 PM"""

@tool
def schedule_meeting(date: str, time: str, title: str, runtime: ToolRuntime[EmailAssistantContext]) -> str:
    """Schedule a meeting.
    
    Args:
        date: Date in YYYY-MM-DD
        time: Time in HH:MM
        title: Meeting title
    """
    return f"✅ Scheduled '{title}' for {date} at {time}"

calendar_tools = [check_calendar, schedule_meeting]

In [21]:
from deepagents import create_deep_agent

agent_v4 = create_deep_agent(
    model="openai:gpt-4o",
    tools=[send_email],
    middleware=[personalized_prompt],
    subagents=[{
        "name": "calendar",
        "description": "Use for calendar/scheduling tasks",
        "system_prompt": "You are a calendar assistant. Help with scheduling.",
        "tools": calendar_tools,
    }],
    context_schema=EmailAssistantContext,
    store=store,
    checkpointer=checkpointer,
    name="agent_v4"
)

### Test Strategy 4: Calendar Subagent

In [22]:
email3 = format_email(
    from_addr="carol@company.com",
    subject="Meeting Request",
    body="Can we meet on March 20th to discuss the project? I'm flexible on timing."
)

config_final = {"configurable": {"thread_id": "final-test"}}

result = agent.invoke(
    {"messages": [("user", f"Respond to: {email3}\n\nCheck calendar for 2024-03-20 and suggest times.")]},
    config_final,
    context=context,
)

for msg in result["messages"]:
    msg.pretty_print()

/Users/sydney_runkle/oss/executive-ai-assistant/tutorial/.venv/lib/python3.13/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=EmailAssistantContext(use...zone='America/New_York'), input_type=EmailAssistantContext])
  return self.__pydantic_serializer__.to_python(


================================ Human Message =================================

Respond to: **From:** carol@company.com
**Subject:** Meeting Request

Can we meet on March 20th to discuss the project? I'm flexible on timing.

Check calendar for 2024-03-20 and suggest times.
================================== Ai Message ==================================
Tool Calls:
  task (call_HKDOLXSEG6GuMeKoDbTMzCSe)
 Call ID: call_HKDOLXSEG6GuMeKoDbTMzCSe
  Args:
    description: Check calendar for availability on 2024-03-20 and suggest three available time slots for a meeting.
    subagent_type: calendar
================================= Tool Message =================================
Name: task

On 2024-03-20, there are two events scheduled:

- **9:00 AM:** Standup (30 min)
- **2:00 PM:** Client Meeting (1 hr)

Here are three available time slots for a meeting:

1. **10:00 AM to 11:00 AM**
2. **11:00 AM to 12:00 PM**
3. **3:00 PM to 4:00 PM**

Let me know if you would like to schedule a meeting d

## Summary: All Four Strategies

This agent combines all context engineering strategies:

1. **Selecting Context**: `@dynamic_prompt` middleware reads from runtime context & store to personalize behavior
2. **Writing Context**: `@after_model` middleware with `interrupt()` learns user preferences from edits using a single LLM call
3. **Summarizing Context**: `SummarizationMiddleware` manages long email chains efficiently
4. **Isolating Context**: Calendar subagent handles scheduling with specialized tools

**Key Components**:
- `create_deep_agent()`: Orchestrates main agent + subagents
- `@dynamic_prompt`: Injects personalized system prompt using `ModelRequest`
- `@after_model`: Reviews tool calls, interrupts for approval, learns from edits using `AgentState` and `Runtime`
- `interrupt()`: Pauses execution for human review (approve/edit)
- Helper functions: `create_interrupt_request()`, `learn_from_edit()` break down complexity
- `SummarizationMiddleware`: Manages conversation length
- `Store`: Persists user preferences across conversations (accessed via `runtime.store`)
- `Checkpointer`: Maintains conversation state
- `Subagents`: Specialized agents with focused tools/prompts

**Simplified Learning Flow**:
1. User edits draft email
2. Single LLM call compares original vs edited
3. LLM extracts tone preference ("casual" or "professional")
4. Preference stored in Store for future personalization

**Middleware Signatures**:
- `@dynamic_prompt` receives `ModelRequest` with access to `request.runtime.store` and `request.runtime.context`
- `@after_model` receives `state: AgentState` and `runtime: Runtime` with access to `runtime.store` and `runtime.context`

**Benefits**:
- ✅ Adaptive behavior based on user preferences
- ✅ Learns from user edits with single LLM call (efficient)
- ✅ Human-in-the-loop for high-stakes operations
- ✅ Efficient handling of long conversations
- ✅ Clean separation of concerns via subagents
- ✅ Helper functions make code readable and maintainable